In [1]:
import os
import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import h5py
import scipy.sparse as sp
import yaml
import time
from tqdm import tqdm
import gget
import scipy
from scipy.stats import zscore
from tqdm import tqdm
from statsmodels.stats.multitest import multipletests
from scipy.spatial.distance import cdist
from scipy.stats import spearmanr

import anndata as an
import scanpy as sc
import scanpy.external as sce
import rapids_singlecell as rsc
import scvi

import cupy as cp
from cuml.manifold import TSNE
from cuml.decomposition import PCA

sc.settings.verbosity = 3

/nfs/turbo/umms-indikar/Cooper/conda_envs/rapids/lib/python3.12/site-packages/docrep/decorators.py:43: SyntaxWarning: 'param_categorical_covariate_keys' is not a valid key!
  doc = func(self, args[0].__doc__, *args[1:], **kwargs)
/nfs/turbo/umms-indikar/Cooper/conda_envs/rapids/lib/python3.12/site-packages/docrep/decorators.py:43: SyntaxWarning: 'param_continuous_covariate_keys' is not a valid key!
  doc = func(self, args[0].__doc__, *args[1:], **kwargs)


# Load deepcycle output

In [2]:
%%time
fpath = "/nfs/turbo/umms-indikar/shared/projects/HSC/pipeline_outputs/DeepCycle/deepcycle_output.h5ad"
adata = sc.read_h5ad(fpath)
adata

CPU times: user 444 ms, sys: 4.17 s, total: 4.61 s
Wall time: 16 s


AnnData object with n_obs × n_vars = 15867 × 21412
    obs: 'batch', 'phase', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'total_counts_mt', 'log1p_total_counts_mt', 'pct_counts_mt', 'total_counts_ribo', 'log1p_total_counts_ribo', 'pct_counts_ribo', 'total_counts_hb', 'log1p_total_counts_hb', 'pct_counts_hb', 'n_counts', 'n_genes', 'n_reads', 'raw_clusters', 'bbknn_clusters', 'harmony_clusters', 'cluster_str', 'barcoded_phase', 'S_score', 'G2M_score', 'dpt_pseudotime', 'dpt_groups', 'dpt_order', 'dpt_order_indices', 'G1_pseudotime', 'G1_order', 'G2M_pseudotime', 'G2M_order', 'mean_pseudotime', 'mean_order', 'nnz', 'velocyto_cell_id', 'cell_id', 'initial_size_unspliced', 'initial_size_spliced', 'initial_size', 'cell_cycle_theta'
    var: 'mt', 'ribo', 'hb', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropou

In [ ]:
%%time
df = adata.to_df()
target_gene = 'NUSAP1'

correlations = df.corrwith(df[target_gene])
totals = df.sum(axis=0)

summary = pd.DataFrame({
    'correlation': correlations,
    'total_expression': totals
}).sort_values('correlation', ascending=False)

summary.head()

In [ ]:
plt.rcParams['figure.dpi'] = 200
plt.rcParams['figure.figsize'] = 4, 4

sc.pl.draw_graph(
    adata,
    ncols=4,
    sort_order=True,
    color=summary.index[:4],
    size=75,
    add_outline=True,
    outline_color=('k', 'k'),
    alpha=1,
    frameon=False,
)

# Load capybara scores

In [ ]:
%%time
# Load Capybara output
fpath_scale = "/nfs/turbo/umms-indikar/shared/projects/HSC/pipeline_outputs/integrated_anndata/capybara/riba_scale.csv"
df = pd.read_csv(fpath_scale)
print(f"{df.shape=}")

# Set index to 'cell_name' for merging
df = df.set_index('cell_name')

# Sanity check: ensure all indices in df exist in adata.obs
common_cells = adata.obs.index.intersection(df.index)
print(f"Merging on {len(common_cells)} shared cells")

# Merge into adata.obs
adata.obs = adata.obs.join(df, how='left')  # 'left' keeps all cells in adata.obs

adata.obs.head()

In [ ]:
plt.rcParams['figure.dpi'] = 200
plt.rcParams['figure.figsize'] = 4, 4

sc.pl.draw_graph(
    adata,
    ncols=4,
    color='phase',
    cmap='hot',
    size=75,
    add_outline=True,
    outline_color=('k', 'k'),
    alpha=1,
    frameon=False,
)

# Pathways

In [ ]:
def get_genes_from_go_path(go_pathname, file_path="../../../resources/GO_Biological_Process_2023.txt"):
    """
    Reads a tab-delimited file where each line starts with a GO pathway name,
    then returns a list of genes in the row where the pathway name matches `go_pathname`.
    """
    with open(file_path, 'r') as f:
        for line in f:
            # Remove newline and split by tab
            parts = line.strip().split('\t')
            # The first column is the pathway name
            if parts[0].strip() == go_pathname.strip():
                # Genes are all non-empty fields after the first two columns
                # (since there are 2 tabs after the pathway name in your file)
                genes = [x for x in parts[2:] if x]
                return genes
    return []

def search_go_pathways(search_str, file_path="../../../resources/GO_Biological_Process_2023.txt"):
    """
    Returns a list of pathway names containing `search_str` (case-insensitive).
    """
    pathways = []
    with open(file_path, 'r') as f:
        for line in f:
            parts = line.strip().split('\t')
            if search_str.lower() in parts[0].lower():
                pathways.append(parts[0])
    return pathways


def pathways_with_genes(gene_list, file_path="../../../resources/GO_Biological_Process_2023.txt"):
    """
    Returns all pathways where every gene in `gene_list` is present in that pathway's gene set.
    """
    results = []
    genes_set = set(gene_list)
    with open(file_path, 'r') as f:
        for line in f:
            parts = line.strip().split('\t')
            pathway = parts[0]
            # Genes start after two tab columns
            pathway_genes = set([x for x in parts[2:] if x])
            if genes_set.issubset(pathway_genes):
                results.append(pathway)
    return results


results = search_go_pathways("arrest")
print(results)
arrest_genes = get_genes_from_go_path(results[0])
arrest_genes = [x for x in arrest_genes if x in adata.var_names]
print(f"{len(arrest_genes)=}")

plt.rcParams['figure.dpi'] = 200
plt.rcParams['figure.figsize'] = 4, 4

sc.pl.draw_graph(
    adata,
    ncols=4,
    color=arrest_genes,
    layer='log_norm',
    cmap='viridis',
    size=75,
    add_outline=True,
    outline_color=('k', 'k'),
    alpha=1,
    frameon=False,
)

In [ ]:
# https://amigo.geneontology.org/amigo/term/GO:0044838 genes 
genes = [
    "ADRB2", "AKT1", "ANGPT1", "BAX", "BCL2", "BCL2L1", "CALCA", "CAV1", "CCNE1", "CD8A",
    "CDC25A", "CDC6", "CDK2", "CDK4", "CDK7", "CDKN1B", "CDKN2A", "CDKN2C", "CGA", "CLCN3",
    "CLU", "COL14A1", "CRH", "CTLA4", "CTNNB1", "DAXX", "DLL4", "DYRK1A", "DYRK1B", "E2F4",
    "EHMT2", "EIF2AK3", "ELF4", "FOXM1", "FSHB", "GAS5", "GATA2", "GDF2", "GSK3A", "GSK3B",
    "HES1", "HEY1", "HGF", "HRAS", "ID1", "IGF1R", "IL1A", "KCNIP3", "KCNK2", "KCNK4", "KNG1",
    "MAPK1", "MAPK14", "MAPK3", "MAPK7", "MCM2", "MIR200B", "MIR302A", "MIR31", "MIR424", "MIR429",
    "MIR503", "MTOR", "MYB", "MYC", "NKRF", "NOLC1", "NOS3", "NOTCH1", "PDGFRB", "PDPN", "PGR",
    "PRKAR2A", "PRKCA", "PTGS2", "RB1", "RBL2", "RRM2B", "RUNX1", "S1PR5", "TEK", "TK2", "TNFAIP3",
    "TP53", "TPPP", "TSC1", "ZEB1", "ZEB2"
]


# 1. Define groups
clusters_of_interest = ["C2", "C3"]
mask_interest = adata.obs['cluster_str'].isin(clusters_of_interest)
mask_other = ~adata.obs['cluster_str'].isin(clusters_of_interest)

# 2. Find shared genes
present_genes = [g for g in genes if g in adata.var_names]
print(f"Genes found in adata: {present_genes}")

# 3. Get expression matrices
if adata.raw is not None:
    X_interest = adata.raw[mask_interest, present_genes].X.toarray()
    X_other = adata.raw[mask_other, present_genes].X.toarray()
else:
    X_interest = adata[mask_interest, present_genes].X.toarray()
    X_other = adata[mask_other, present_genes].X.toarray()

# 4. Compute mean expression (+ pseudocount for log transform)
mean_interest = np.mean(X_interest, axis=0) + 1e-6
mean_other = np.mean(X_other, axis=0) + 1e-6

# 5. Calculate log2 fold change
log2fc = np.log2(mean_interest / mean_other)

# 6. Threshold: e.g., log2FC > 1 (upregulated at least 2-fold)
upregulated = [gene for gene, fc in zip(present_genes, log2fc) if fc > 1]

# 7. Output
fc_df = pd.DataFrame({
    "gene": present_genes,
    "log2fc": log2fc
}).sort_values("log2fc", ascending=False)

print("Genes upregulated in C2/C3 vs other clusters (log2FC > 1):")
print(fc_df[fc_df["log2fc"] > 0.85])

# If you just want the list:
print(upregulated)

plt.rcParams['figure.dpi'] = 200
plt.rcParams['figure.figsize'] = 4, 4

sc.pl.draw_graph(
    adata,
    ncols=3,
    color=fc_df[fc_df["log2fc"] > 0.85]['gene'].to_list(),
    layer='log_norm',
    cmap='viridis',
    size=75,
    add_outline=True,
    outline_color=('k', 'k'),
    alpha=1,
    frameon=False,
)


In [ ]:
sc.tl.score_genes(
    adata,
    gene_list=genes,
    score_name='G0 score',
    layer='log_norm',
    ctrl_size=200,
)

plt.rcParams['figure.dpi'] = 200
plt.rcParams['figure.figsize'] = 4.5, 4

sc.pl.draw_graph(
    adata,
    ncols=3,
    color='G0 score',
    layer='log_norm',
    cmap='plasma',
    size=75,
    add_outline=True,
    outline_color=('k', 'k'),
    alpha=1,
    frameon=False,
)

In [ ]:
plt.figure(figsize=(3, 2))

sns.boxplot(
    data=adata.obs,
    x='cluster_str',
    y='G0 score',
    hue='cluster_str',
    showfliers=False,
    width=0.5,
    linewidth=1,
    linecolor='k',
    capprops={'lw' : 0},
)

sns.despine()
plt.xlabel("")
plt.tight_layout()
plt.show()

In [ ]:
plt.rcParams['figure.dpi'] = 200
plt.rcParams['figure.figsize'] = 4, 4

frxn_cols = [col for col in df.columns if col.startswith("frxn_")]

sc.pl.draw_graph(
    adata,
    ncols=4,
    color=frxn_cols,
    cmap='hot',
    size=75,
    add_outline=True,
    outline_color=('k', 'k'),
    title=[x.replace("frxn_", "") for x in frxn_cols],
    alpha=1,
    frameon=False,
)

In [ ]:
plt.rcParams['figure.dpi'] = 200
plt.rcParams['figure.figsize'] = 4, 4

genes = ["MXD4",]

"""
MXD4 is a MYC antagonist known to increase the fraction of cells in the G0/G1 phase in hematopoietic differentiation, 
and could be a master regulator of entry into the quiescence-like state.
"""
sc.pl.draw_graph(
    adata,
    ncols=4,
    color=genes,
    size=75,
    add_outline=True,
    outline_color=('k', 'k'),
    alpha=1,
    frameon=False,
)

In [ ]:
# break

In [ ]:
plt.figure(figsize=(3, 3))
sns.boxplot(
    data=adata.obs,
    x='cluster_str',
    y='cell_cycle_theta',
    hue='cluster_str',
    showfliers=False,
    width=0.5,
    linewidth=1,
    linecolor='k',
    capprops={'lw' : 0},
)

plt.ylabel(r'Cell Cycle $\theta$', fontsize=10)
plt.xlabel('Cluster', fontsize=10)
sns.despine()
plt.tight_layout()
plt.show()

In [ ]:
# Copy and melt
df = adata.obs.copy()
frxn_cols = [col for col in df.columns if col.startswith("frxn_")]
print(frxn_cols)
df_melted = df.melt(
    id_vars=["cluster_str"],
    value_vars=frxn_cols,
    var_name="Cell Type",
    value_name="Fraction"
)

df_melted['Cell Type'] = df_melted['Cell Type'].str.replace('frxn_', '')
df_melted = df_melted.sort_values('Cell Type', ascending=False)
print(df_melted['Cell Type'].unique())

plt.rcParams['figure.dpi'] = 300
plt.rcParams['figure.figsize'] = 4, 4

palette = {
    'G0':  "#999999",  # neutral grey for quiescence
    'G1':  "#D7263D",  # FUCCI red (e.g., Cdt1-mCherry)
    'S':   "#F4A261",  # orange (transitional)
    'G2M': "#1E88E5",  # FUCCI blue (e.g., Geminin-CFP)
}


# Plot with FacetGrid
g = sns.FacetGrid(
    df_melted,
    col="Cell Type",
    col_wrap=5,
    sharey=True,
    height=2,
    aspect=0.8,
)

g.map_dataframe(
    sns.boxplot,
    x="cluster_str",
    y="Fraction",
    hue='Cell Type',
    order=sorted(df["cluster_str"].unique()),
    width=0.45,
    linecolor='k',
    palette=palette,
    capprops={'linewidth': 0},  
    showfliers=False,
    linewidth=0.8
)

g.set_axis_labels("", "proportion")
g.set_titles(col_template="{col_name}", size=8)
for ax in g.axes.flat:
    ax.tick_params(axis='x', rotation=0, labelsize=8)
    ax.grid(axis='y', linestyle='--', alpha=0.3)

g.fig.subplots_adjust(wspace=0.2)

plt.tight_layout()
sns.despine(trim=True)
plt.show()

In [ ]:
plt.rcParams['figure.dpi'] = 200
plt.rcParams['figure.figsize'] = 4, 4

adata.obs['riba'] = adata.obs[frxn_cols].idxmax(axis=1)

sc.pl.draw_graph(
    adata,
    ncols=3,
    color=['cell_cycle_theta', 'phase', 'riba'],
    cmap='vlag',
    size=75,
    add_outline=True,
    outline_color=('k', 'k'),
    alpha=1,
    frameon=False,
)

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Crosstab (absolute counts)
ct = pd.crosstab(adata.obs['riba'], adata.obs['phase'])

# Plot
plt.figure(figsize=(4, 3))
sns.heatmap(
    ct, annot=True, fmt='d', cmap='Blues',
    cbar_kws={'label': 'Cell Count'},
    linewidths=0.5, linecolor='gray'
)
plt.xlabel('Cell Cycle Phase', fontsize=10)
plt.ylabel('CAPYBARA Assignment', fontsize=10)
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Crosstab (absolute counts)
ct = pd.crosstab(adata.obs['riba'], adata.obs['cluster_str'])

# Plot
plt.figure(figsize=(4.5, 3))
sns.heatmap(
    ct, annot=True, fmt='d', cmap='Greens',
    cbar_kws={'label': 'Cell Count'},
    linewidths=0.5, linecolor='gray'
)
plt.xlabel('Cluster', fontsize=10)
plt.ylabel('CAPYBARA Assignment', fontsize=10)
plt.tight_layout()
plt.show()


In [ ]:
pdf = adata.obs.copy()
pdf = pdf[pdf['cluster_str'].isin(['C4', 'C5'])]

plt.rcParams['figure.dpi'] = 200
plt.rcParams['figure.figsize'] = 2, 2.5

sns.boxplot(
    data=pdf,
    x='phase',
    y='cell_cycle_theta',
    hue='phase',
    order=['G1', 'S', 'G2M'],
    showfliers=False,
    palette=adata.uns['phase_colors'],
    capprops={'lw' : 0},
    linecolor='k',
    width=0.7
)
plt.ylabel(r"$\theta$")
plt.xlabel("")
sns.despine()


In [ ]:
# get higest expressed S phase genes
cell_cycle_genes = [x.strip() for x in open('../../../resources/regev_lab_cell_cycle_genes.txt')]
s_genes = [x for x in cell_cycle_genes[:43] if x in adata.var_names]
g2m_genes = [x for x in cell_cycle_genes[43:] if x in adata.var_names]

print(f"N S genes: {len(s_genes)}")
print(f"N G2M genes: {len(g2m_genes)}")

In [ ]:
# Filter cells
threshold = 0.55
mask = (adata.obs['cluster_str'].isin(['C4', 'C5']) & (adata.obs['cell_cycle_theta'] < threshold))
subset = adata[mask, s_genes].copy()

# Get top 9 expressed S-phase genes
gdf = subset.to_df()
top_genes = gdf.mean().sort_values(ascending=False).head(9)

# Plot
fig, axes = plt.subplots(3, 3, figsize=(6, 6), dpi=150)
axes = axes.flatten()

for i, gene in enumerate(top_genes.index):
    sns.boxplot(
        data=subset.obs.assign(expr=gdf[gene].values),
        x='phase',
        y='expr',
        ax=axes[i],
        hue='phase',
        order=['G1', 'S', 'G2M'],
        showfliers=False,
        palette=adata.uns['phase_colors'],
        capprops={'lw' : 0},
        linecolor='k',
        width=0.7
    )
    axes[i].set_title(gene)
    axes[i].set_ylabel("Expression")
    axes[i].set_xlabel("")
    # break

sns.despine()
plt.tight_layout()
plt.suptitle('S Phase Genes', y=1.1)
plt.show()


In [ ]:
# Filter cells
threshold = 0.55
mask = (adata.obs['cluster_str'].isin(['C4', 'C5']) & (adata.obs['cell_cycle_theta'] < threshold))
subset = adata[mask, g2m_genes].copy()

# Get top 9 expressed S-phase genes
gdf = subset.to_df()
top_genes = gdf.mean().sort_values(ascending=False).head(9)

# Plot
fig, axes = plt.subplots(3, 3, figsize=(6, 6), dpi=150)
axes = axes.flatten()

for i, gene in enumerate(top_genes.index):
    sns.boxplot(
        data=subset.obs.assign(expr=gdf[gene].values),
        x='phase',
        y='expr',
        ax=axes[i],
        hue='phase',
        order=['G1', 'S', 'G2M'],
        showfliers=False,
        palette=adata.uns['phase_colors'],
        capprops={'lw' : 0},
        linecolor='k',
        width=0.7
    )
    axes[i].set_title(gene)
    axes[i].set_ylabel("Expression")
    axes[i].set_xlabel("")
    # break

sns.despine()
plt.tight_layout()
plt.suptitle('G2/M Phase Genes', y=1.1)
plt.show()


In [ ]:
# Filter cells
threshold = 0.55
mask = (adata.obs['cluster_str'].isin(['C4', 'C5']) & (adata.obs['cell_cycle_theta'] < threshold))
subset = adata[mask, s_genes].copy()

# Get top 9 expressed S-phase genes
gdf = subset.to_df()
top_genes = gdf.mean().sort_values(ascending=False).head(9)

# Plot
fig, axes = plt.subplots(3, 3, figsize=(6, 6), dpi=150)
axes = axes.flatten()

for i, gene in enumerate(top_genes.index):
    expr = gdf[gene].values
    theta = subset.obs['cell_cycle_theta'].values

    # Drop zero-expression cells
    nonzero_mask = expr > 0
    expr = expr[nonzero_mask]
    theta = theta[nonzero_mask]

    sns.scatterplot(
        x=theta,
        y=expr,
        alpha=0.1,
        s=5,
        color='black',
        ax=axes[i]
    )
  
    # Bin and smooth
    df = pd.DataFrame({'theta': theta, 'expr': expr})
    df = df.sort_values('theta')
    bins = np.linspace(0, 1, 25)
    df['bin'] = np.digitize(df['theta'], bins)
    df_smooth = df.groupby('bin')[['theta', 'expr']].mean().dropna()

    axes[i].plot(df_smooth['theta'], df_smooth['expr'], color='red', lw=1.5, marker=".")
    axes[i].set_title(gene)
    axes[i].set_xlabel(r"$\theta$")
    axes[i].set_ylabel("Expression")

sns.despine()
plt.tight_layout()
plt.suptitle('S Phase Genes', y=1.1)
plt.show()


In [ ]:
# Filter cells
threshold = 0.55
mask = (adata.obs['cluster_str'].isin(['C4', 'C5']) & (adata.obs['cell_cycle_theta'] < threshold))
subset = adata[mask, g2m_genes].copy()

# Get top 9 expressed S-phase genes
gdf = subset.to_df()
top_genes = gdf.mean().sort_values(ascending=False).head(9)

# Plot
fig, axes = plt.subplots(3, 3, figsize=(6, 6), dpi=150)
axes = axes.flatten()

for i, gene in enumerate(top_genes.index):
    expr = gdf[gene].values
    theta = subset.obs['cell_cycle_theta'].values

    # Drop zero-expression cells
    nonzero_mask = expr > 0
    expr = expr[nonzero_mask]
    theta = theta[nonzero_mask]

    sns.scatterplot(
        x=theta,
        y=expr,
        alpha=0.1,
        s=5,
        color='black',
        ax=axes[i]
    )
  
    # Bin and smooth
    df = pd.DataFrame({'theta': theta, 'expr': expr})
    df = df.sort_values('theta')
    bins = np.linspace(0, 1, 25)
    df['bin'] = np.digitize(df['theta'], bins)
    df_smooth = df.groupby('bin')[['theta', 'expr']].mean().dropna()

    axes[i].plot(df_smooth['theta'], df_smooth['expr'], color='red', lw=1.5, marker=".")
    axes[i].set_title(gene)
    axes[i].set_xlabel(r"$\theta$")
    axes[i].set_ylabel("Expression")

sns.despine()
plt.tight_layout()
plt.suptitle('G2/M Phase Genes', y=1.1)
plt.show()

# Quiescence

In [ ]:
# Read Markers
fpath = "../../../resources/quiescence_markers.csv"
df = pd.read_csv(fpath)
print(f"raw {df.shape=}")
df = df[df['gene'].isin(adata.var.index.to_list())]
print(f"filtered {df.shape=}")
df.head()

In [ ]:
new_columns = []
for source, group in df.groupby('source'):
    gene_list = group['gene'].to_list()

    print(f"{source}:")
    print(f"\t{len(gene_list)} genes")
    column_name = f'{source}_G0_score'

    sc.tl.score_genes(
        adata, 
        gene_list, 
        ctrl_as_ref=True,
        ctrl_size=len(gene_list),
        gene_pool=None, 
        n_bins=25,
        score_name=f'{source}_G0_score',
    )
    print(f'\tAdded column: {column_name}')
    new_columns.append(column_name)

print(new_columns)
print('done!')

plt.rcParams['figure.dpi'] = 200
plt.rcParams['figure.figsize'] = 4, 4

sc.pl.draw_graph(
    adata,
    ncols=5,
    color=new_columns,
    cmap='bwr',
    size=75,
    add_outline=True,
    outline_color=('k', 'k'),
    alpha=1,
    frameon=False,
)

In [ ]:
plt.rcParams['figure.dpi'] = 200
plt.rcParams['figure.figsize'] = 4, 4

sc.pl.draw_graph(
    adata,
    ncols=2,
    sort_order=True,
    color='G1_to_G0_up_G0_score',
    cmap='bwr',
    size=75,
    add_outline=True,
    outline_color=('k', 'k'),
    alpha=1,
    frameon=False,
)

In [ ]:
pdf = adata.obs.copy()

plt.rcParams['figure.dpi'] = 200
plt.rcParams['figure.figsize'] = 2.5, 2.5

sns.boxplot(
    data=pdf,
    x='cluster_str',
    y='G1_to_G0_up_G0_score',
    showfliers=False,
    capprops={'lw' : 0},
    linecolor='k',
    hue='cluster_str',
    width=0.7
)

plt.grid(axis='y', linestyle='--', color='grey', linewidth=0.5, alpha=0.4)
plt.axhline(y=0, lw=1, c='grey', zorder=0)
plt.ylabel(r"$\text{G}_0$ enrichment score")
plt.xlabel("")
sns.despine()


In [ ]:
pdf = adata.obs.copy()

plt.rcParams['figure.dpi'] = 200
plt.rcParams['figure.figsize'] = 2, 2.5

sns.boxplot(
    data=pdf,
    x='phase',
    y='G1_to_G0_up_G0_score',
    hue='phase',
    order=['G1', 'S', 'G2M'],
    showfliers=False,
    palette=adata.uns['phase_colors'],
    capprops={'lw' : 0},
    legend=False,
    linecolor='k',
    width=0.7
)

plt.grid(axis='y', linestyle='--', color='grey', linewidth=0.5, alpha=0.4)
plt.axhline(y=0, lw=1, c='grey', zorder=0)
plt.ylabel(r"$\text{G}_0$ enrichment score")
plt.xlabel("")
sns.despine()


In [ ]:
sc.tl.rank_genes_groups(
    adata, 
    groupby='cluster_str',
    method='wilcoxon',
    layer='log_norm',
    use_raw=False,
    pts=True,
)

deg = sc.get.rank_genes_groups_df(
    adata, 
    group=None,
    pval_cutoff=0.05,
)


plt.rcParams['figure.dpi'] = 200
plt.rcParams['figure.figsize'] = 6, 3
sc.pl.rank_genes_groups(
    adata,
    n_genes=35,
    sharey=False,
    ncols=3,
)

deg.head()

In [ ]:
pdf = deg.copy()
pdf = pdf[pdf['group'] == 'C3']
print(f"raw {pdf.shape}")
pdf = pdf[pdf['names'].isin(df[df['source'] == 'G1_to_G0_up']['gene'].to_list())]
print(f"filtered {pdf.shape}")
pdf = pdf.sort_values(by='logfoldchanges', ascending=False)
pdf.head()

In [ ]:
plt.rcParams['figure.dpi'] = 200
plt.rcParams['figure.figsize'] = 4, 4

sc.pl.draw_graph(
    adata,
    ncols=3,
    layer='log_norm',
    color=pdf['names'].head(9),
    cmap='viridis',
    size=75,
    add_outline=True,
    outline_color=('k', 'k'),
    alpha=1,
    frameon=False,
    show=False,
)

fig = plt.gcf()
fig.suptitle(r"$\text{G}_0$ marker expression")

In [ ]:
plt.rcParams['figure.dpi'] = 200
plt.rcParams['figure.figsize'] = 4, 4

sc.pl.draw_graph(
    adata,
    ncols=3,
    color=pdf['names'].head(9),
    cmap='viridis',
    size=75,
    add_outline=True,
    outline_color=('k', 'k'),
    alpha=1,
    frameon=False,
    show=False,
)

fig = plt.gcf()
fig.suptitle(r"$\text{G}_0$ marker expression")

In [ ]:
g0 = df[df['source'].str.contains('up')].copy()

for group_name, group in g0.groupby('source'):

    pdf = deg.copy()
    pdf = pdf[pdf['group'] == 'C3']
    print(f"{group_name} marker set:")
    print(f"\traw {pdf.shape=}")
    
    pdf = pdf[pdf['names'].isin(group['gene'].unique())]
    print(f"\tfiltered {pdf.shape}")
    pdf = pdf.sort_values(by='logfoldchanges', ascending=False)

    plt.rcParams['figure.dpi'] = 200
    plt.rcParams['figure.figsize'] = 4, 4
    
    sc.pl.draw_graph(
        adata,
        ncols=4,
        layer='log_norm',
        color=pdf['names'].head(16),
        cmap='viridis',
        size=75,
        add_outline=True,
        outline_color=('k', 'k'),
        alpha=1,
        frameon=False,
        show=False,
    )
    
    fig = plt.gcf()
    fig.suptitle(f"G0 ({group_name})")
    plt.show()



In [ ]:
# plt.rcParams['figure.dpi'] = 200
# plt.rcParams['figure.figsize'] = 4, 4

# sc.pl.draw_graph(
#     adata,
#     ncols=3,
#     color=pdf['names'].head(9),
#     cmap='viridis',
#     size=75,
#     add_outline=True,
#     outline_color=('k', 'k'),
#     alpha=1,
#     frameon=False,
#     show=False,
# )

# fig = plt.gcf()
# fig.suptitle(r"$\text{G}_0$ marker expression")